In [ ]:
%pip install -q keras-hub

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import numpy as np
import tensorflow as tf
import warnings
warnings.filterwarnings("ignore")


# Check TensorFlow and GPU
print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

# Dataset paths
BASE_PATH = "/kaggle/input/notebooks/shazahanybakr/dataset"
DATA_ROOT = os.path.join(BASE_PATH, "processed_data")

# Check processed data
print("\nDataset exists:", os.path.exists(DATA_ROOT))

for split in ["train", "val", "test"]:
    split_path = os.path.join(DATA_ROOT, split)

    if os.path.exists(split_path):
        print(f"\n{split.upper()}")
        print("Images:", len(os.listdir(os.path.join(split_path, "images"))))
        print("Masks:", len(os.listdir(os.path.join(split_path, "masks"))))
    else:
        print(f"{split} folder not found!")

TensorFlow: 2.21.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

Dataset exists: True

TRAIN
Images: 7470
Masks: 7470

VAL
Images: 1602
Masks: 1602

TEST
Images: 1602
Masks: 1602


## **Preparing image paths and masks**

In [ ]:

# Match every image with its corresponding mask

def get_image_mask_pairs(split):

    image_dir = os.path.join(DATA_ROOT, split, "images")
    mask_dir = os.path.join(DATA_ROOT, split, "masks")

    image_paths = sorted(glob(os.path.join(image_dir, "*.jpg")))
    mask_paths = sorted(glob(os.path.join(mask_dir, "*.png")))

    # Create a mapping between mask names and paths
    mask_map = {}

    for mask_path in mask_paths:

        mask_name = os.path.basename(mask_path)
        mask_stem = os.path.splitext(mask_name)[0]

        # Support both naming patterns:
        # filename.png OR filename_m.png
        if mask_stem.endswith("_m"):
            mask_stem = mask_stem[:-2]

        mask_map[mask_stem] = mask_path

    image_list = []
    mask_list = []

    for image_path in image_paths:

        image_name = os.path.splitext(
            os.path.basename(image_path)
        )[0]

        if image_name not in mask_map:
            raise ValueError(f"Missing mask for: {image_name}")

        image_list.append(image_path)
        mask_list.append(mask_map[image_name])

    if len(image_list) != len(mask_list):
        raise ValueError(f"Image-mask mismatch in {split}")

    print(f"{split.upper()}: {len(image_list)} matched pairs")

    return image_list, mask_list


# Create pairs for all splits

train_images, train_masks = get_image_mask_pairs("train")
val_images, val_masks = get_image_mask_pairs("val")
test_images, test_masks = get_image_mask_pairs("test")


TRAIN: 7470 matched pairs
VAL: 1602 matched pairs
TEST: 1602 matched pairs


### **TensorFlow Data Pipeline**

In [ ]:
# Read image and mask
def load_image_mask(image_path, mask_path):

    # Read RGB image
    image = tf.io.read_file(image_path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.convert_image_dtype(image, tf.float32)

    # Read segmentation mask
    mask = tf.io.read_file(mask_path)
    mask = tf.io.decode_png(mask, channels=1)
    mask = tf.cast(tf.squeeze(mask, axis=-1), tf.int32)

    # Verify dimensions
    tf.debugging.assert_equal(
        tf.shape(image)[:2],
        tf.shape(mask),
        message="Image and mask dimensions do not match!"
    )

    # Verify class IDs
    tf.debugging.assert_greater_equal(mask, 0)
    tf.debugging.assert_less(mask, NUM_CLASSES)

    image.set_shape([IMG_SIZE, IMG_SIZE, 3])
    mask.set_shape([IMG_SIZE, IMG_SIZE])

    return image, mask


# Apply paired geometric augmentation
def augment_pair(image, mask):

    mask = mask[..., tf.newaxis]

    # Horizontal flip
    image, mask = tf.cond(
        tf.random.uniform(()) < 0.5,
        lambda: (
            tf.image.flip_left_right(image),
            tf.image.flip_left_right(mask)
        ),
        lambda: (image, mask)
    )

    # Vertical flip
    image, mask = tf.cond(
        tf.random.uniform(()) < 0.5,
        lambda: (
            tf.image.flip_up_down(image),
            tf.image.flip_up_down(mask)
        ),
        lambda: (image, mask)
    )

    # Random 90-degree rotation
    k = tf.random.uniform([], minval=0, maxval=4, dtype=tf.int32)

    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)

    mask = tf.squeeze(mask, axis=-1)

    # Brightness and contrast (image only)
    image = tf.image.random_brightness(image, max_delta=0.15)
    image = tf.image.random_contrast(
        image, lower=0.85, upper=1.15
    )

    image = tf.clip_by_value(image, 0.0, 1.0)

    return image, mask


# Normalize images
def normalize_image(image, mask):

    image = (image - IMG_MEAN) / IMG_STD

    return image, mask


# Build dataset
def make_dataset(image_paths, mask_paths, training=False):

    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, mask_paths)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(image_paths), 2000),
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image_mask,
        num_parallel_calls=AUTOTUNE
    )

    if training:
        dataset = dataset.map(
            augment_pair,
            num_parallel_calls=AUTOTUNE
        )

    dataset = dataset.map(
        normalize_image,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(
        BATCH_SIZE,
        drop_remainder=training
    )

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


# Create TensorFlow datasets
train_ds = make_dataset(
    train_images, train_masks, training=True
)

val_ds = make_dataset(
    val_images, val_masks, training=False
)

test_ds = make_dataset(
    test_images, test_masks, training=False
)

print("TensorFlow datasets created successfully")


TensorFlow datasets created successfully
